<a href="https://colab.research.google.com/github/linhkid/pallas-forge/blob/main/notebooks/02_fused_rmsnorm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Fused RMSNorm + Residual Addition

A **memory-bound** kernel demonstrating:

- **Kernel fusion**: combining RMSNorm + residual add into one pass
- **VPU vs MXU**: this kernel is VPU-bound (elementwise ops), not MXU-bound
- **When fusion wins**: reducing HBM round-trips beats XLA in some cases

### What is RMSNorm?

RMSNorm (Root Mean Square Normalization) is used in modern transformers (LLaMA, Gemma):

```
RMSNorm(x) = x / sqrt(mean(x^2) + eps) * weight
```

In practice, it's always paired with a residual connection:
```
new_residual = x + residual
output = RMSNorm(new_residual) * weight
```

Fusing these avoids writing `new_residual` to HBM and re-reading it.

In [1]:
# ============================================================
# Colab / TPU Setup — run this cell first!
# ============================================================
# On Colab: Runtime > Change runtime type > TPU
# Skip this cell if running locally with pallas-forge already installed.
# ============================================================

import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[tpu]", "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ]
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pallas-forge[viz] @ git+https://github.com/linhkid/pallas-forge.git@v0.1.2",
        ]
    )
    print("Colab setup complete!")
else:
    print("Running locally — skipping Colab install.")

Colab setup complete!


In [2]:
import jax
import jax.numpy as jnp

from pallas_forge import fused_rmsnorm_residual
from pallas_forge.kernels.rmsnorm import rmsnorm_reference

## 1. Reference implementation

First, let's see the unfused version (what XLA would compile).

In [3]:
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

# Typical transformer dimensions
batch, seq_len, dim = 2, 8, 128

x = jax.random.normal(k1, (batch, seq_len, dim), dtype=jnp.float32)
residual = jax.random.normal(k2, (batch, seq_len, dim), dtype=jnp.float32)
weight = jax.random.normal(k3, (dim,), dtype=jnp.float32)

# Unfused (2 separate ops)
new_residual_ref = x + residual
normed_ref = rmsnorm_reference(new_residual_ref, weight)

print(f"Input shape: {x.shape}")
print(f"Weight shape: {weight.shape}")
print(f"Output shape: {normed_ref.shape}")

Input shape: (2, 8, 128)
Weight shape: (128,)
Output shape: (2, 8, 128)


## 2. Fused Pallas kernel

The fused kernel does both operations in a single pass. Each kernel instance
processes one token's entire hidden dimension.

In [4]:
# Fused (single kernel pass)
normed_fused, new_residual_fused = fused_rmsnorm_residual(x, residual, weight)

# Verify correctness
res_error = float(jnp.max(jnp.abs(new_residual_fused - new_residual_ref)))
norm_error = float(jnp.max(jnp.abs(normed_fused - normed_ref)))

print(f"Residual add error: {res_error:.2e}")
print(f"RMSNorm error:      {norm_error:.2e}")
print(f"Residual correct:   {bool(jnp.allclose(new_residual_fused, new_residual_ref, atol=1e-4))}")
print(f"Norm correct:       {bool(jnp.allclose(normed_fused, normed_ref, atol=1e-4))}")

Residual add error: 0.00e+00
RMSNorm error:      0.00e+00
Residual correct:   True
Norm correct:       True


## 3. Why fusion matters — HBM traffic analysis

Let's count the bytes moved in each approach:

In [5]:
tokens = batch * seq_len
elem_bytes = 4  # float32

# Unfused: 2 separate kernel launches
# Op 1 (residual add): read x + residual, write new_residual
unfused_op1 = (tokens * dim * 2 + tokens * dim) * elem_bytes
# Op 2 (rmsnorm): read new_residual + weight, write output
unfused_op2 = (tokens * dim + dim + tokens * dim) * elem_bytes
unfused_total = unfused_op1 + unfused_op2

# Fused: single kernel launch
# read x + residual + weight, write output + new_residual
fused_total = (tokens * dim * 2 + dim + tokens * dim * 2) * elem_bytes

savings = unfused_total - fused_total
savings_pct = savings / unfused_total * 100

print(f"Unfused HBM traffic: {unfused_total / 1024:.1f} KB")
print(f"  - Residual add:    {unfused_op1 / 1024:.1f} KB")
print(f"  - RMSNorm:         {unfused_op2 / 1024:.1f} KB")
print(f"Fused HBM traffic:   {fused_total / 1024:.1f} KB")
print(f"Savings:             {savings / 1024:.1f} KB ({savings_pct:.1f}%)")
print("\nThe fused kernel avoids writing & re-reading the intermediate residual.")

Unfused HBM traffic: 40.5 KB
  - Residual add:    24.0 KB
  - RMSNorm:         16.5 KB
Fused HBM traffic:   32.5 KB
Savings:             8.0 KB (19.8%)

The fused kernel avoids writing & re-reading the intermediate residual.


## 4. bfloat16 support

In [6]:
x_bf16 = x.astype(jnp.bfloat16)
res_bf16 = residual.astype(jnp.bfloat16)
w_bf16 = weight.astype(jnp.bfloat16)

normed_bf16, new_res_bf16 = fused_rmsnorm_residual(x_bf16, res_bf16, w_bf16)

print(f"Output dtype: {normed_bf16.dtype}")
print(f"Internal computation in float32, cast back to {normed_bf16.dtype}")

Output dtype: bfloat16
Internal computation in float32, cast back to bfloat16


## Key takeaways

1. **Kernel fusion** reduces HBM traffic by eliminating intermediate writes
2. **RMSNorm is VPU-bound** — it uses elementwise ops, not the MXU (systolic array)
3. On TPU, the block_size parameter would control VMEM pressure, but for RMSNorm the best strategy is usually to process the full hidden dimension per token
4. The fusion benefit grows with hidden dimension size (more bytes saved)

Next: [03_swiglu_geglu.ipynb](03_swiglu_geglu.ipynb) — a compute-bound fused activation.